# CUDA Fundamentals

Establish the CUDA execution concepts needed for later GPU experiments.

## Objectives

Detect CUDA safely and frame an experiment around host/device execution and synchronization.

## Background

CUDA launches work on a GPU execution hierarchy; asynchronous execution means correct timing requires deliberate synchronization.

## Prediction

A CUDA operation has at least three distinguishable costs:

1. host-side launch overhead;
2. GPU execution time;
3. host-side waiting caused by synchronization.

Because CUDA launches are normally asynchronous, measuring only the Python call duration should initially report mostly host-side enqueue overhead rather than completed GPU work.

The first CUDA operation may be slower than later operations because the CUDA context, memory allocator, and kernel machinery may require one-time initialization.

For very small tensors, fixed launch and synchronization overhead should dominate. As tensor size increases, GPU execution time should become a larger fraction of total elapsed time.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [4]:
from pprint import pprint

from common.cuda import detect_cuda


cuda_info = detect_cuda()
pprint(cuda_info)

if not cuda_info.torch_installed:
    raise RuntimeError(
        "PyTorch is not installed in this environment. "
        "CUDA experiments cannot continue."
    )

if not cuda_info.available:
    raise RuntimeError(
        "PyTorch is installed, but CUDA is not available. "
        f"Detection error: {cuda_info.error!r}"
    )

CudaInfo(torch_installed=True,
         available=True,
         device_count=1,
         device_names=('NVIDIA GB10',),
         torch_version='2.13.0+cu130',
         cuda_version='13.0',
         error=None)


### CUDA runtime and device properties

Before measuring execution, inspect the CUDA runtime exposed through PyTorch and the properties of the selected device.

These values are environment facts. They describe the software and hardware visible to this process but do not yet measure performance.

In [5]:
import torch


device_index = torch.cuda.current_device()
device = torch.device(f"cuda:{device_index}")
properties = torch.cuda.get_device_properties(device_index)

runtime_info = {
    "torch_version": torch.__version__,
    "torch_cuda_build_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count(),
    "current_device_index": device_index,
    "current_device_name": torch.cuda.get_device_name(device_index),
    "compute_capability": (
        properties.major,
        properties.minor,
    ),
    "multiprocessor_count": properties.multi_processor_count,
    "total_device_memory_bytes": properties.total_memory,
    "total_device_memory_gib": properties.total_memory / 1024**3,
}

pprint(runtime_info)

{'compute_capability': (12, 1),
 'cuda_available': True,
 'current_device_index': 0,
 'current_device_name': 'NVIDIA GB10',
 'device_count': 1,
 'multiprocessor_count': 48,
 'torch_cuda_build_version': '13.0',
 'torch_version': '2.13.0+cu130',
 'total_device_memory_bytes': 130663002112,
 'total_device_memory_gib': 121.68940353393555}


In [6]:
values = torch.arange(
    8,
    dtype=torch.float32,
    device=device,
)

result = values * 2.0 + 1.0

print(f"values device: {values.device}")
print(f"result device: {result.device}")
print(f"result: {result.cpu().tolist()}")

expected = [1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0]
assert result.cpu().tolist() == expected

values device: cuda:0
result device: cuda:0
result: [1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0]


### Asynchronous launch and synchronization

A CUDA tensor operation invoked from Python is normally enqueued onto a CUDA stream. The Python call may return before the GPU has completed the operation.

Three timings are compared:

- **enqueue time** measures how long the Python call takes without waiting explicitly;
- **synchronized host time** includes the host-side wait for completion;
- **CUDA event time** measures elapsed time on the CUDA stream around the operation.

The operation is warmed up before measurement so that this comparison focuses on steady-state execution rather than first-use initialization.

In [ ]:
import time

import pandas as pd


ELEMENT_COUNT = 16_777_216
DTYPE = torch.float32

input_values = torch.linspace(
    0.0,
    1.0,
    ELEMENT_COUNT,
    dtype=DTYPE,
    device=device,
)

output_values = torch.empty_like(input_values)


def cuda_multiply_add() -> None:
    torch.add(
        input_values * 1.000_001,
        0.125,
        out=output_values,
    )


for _ in range(10):
    cuda_multiply_add()

torch.cuda.synchronize(device)

sample_indices = torch.tensor(
    [0, ELEMENT_COUNT // 2, ELEMENT_COUNT - 1],
    device=device,
)

sample_input = input_values[sample_indices].cpu()
sample_output = output_values[sample_indices].cpu()
sample_expected = sample_input * 1.000_001 + 0.125

print(f"Element count: {ELEMENT_COUNT:,}")
print(
    f"Tensor size: {input_values.numel() * input_values.element_size() / 1024**2:.1f} MiB"
)
print(f"Input samples: {sample_input.tolist()}")
print(f"Output samples: {sample_output.tolist()}")

torch.testing.assert_close(sample_output, sample_expected)

Element count: 16,777,216
Tensor size: 64.0 MiB
Input samples: [0.0, 0.5, 1.0]
Output samples: [0.125, 0.6250004768371582, 1.1250009536743164]


In [ ]:
REPETITIONS = 100

timing_rows = []

for repetition in range(REPETITIONS):
    torch.cuda.synchronize(device)

    start_ns = time.perf_counter_ns()
    cuda_multiply_add()
    enqueue_end_ns = time.perf_counter_ns()

    torch.cuda.synchronize(device)
    synchronized_end_ns = time.perf_counter_ns()

    timing_rows.append(
        {
            "repetition": repetition,
            "enqueue_us": (enqueue_end_ns - start_ns) / 1_000,
            "synchronized_us": (synchronized_end_ns - start_ns) / 1_000,
        }
    )

host_timings = pd.DataFrame(timing_rows)

host_timings.describe(percentiles=[0.5, 0.9, 0.99])[["enqueue_us", "synchronized_us"]]

,enqueue_us,synchronized_us
count,100.000000,100.00000
mean,9.858280,1247.90552
std,13.864416,53.67291
min,7.488000,1213.58800
50%,8.040000,1225.52400
90%,10.081600,1333.99280
99%,17.115840,1454.91827
max,146.592000,1518.40400


In [ ]:
event_rows = []

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

for repetition in range(REPETITIONS):
    start_event.record()
    cuda_multiply_add()
    end_event.record()

    end_event.synchronize()

    event_rows.append(
        {
            "repetition": repetition,
            "cuda_event_us": start_event.elapsed_time(end_event) * 1_000,
        }
    )

event_timings = pd.DataFrame(event_rows)

event_timings.describe(percentiles=[0.5, 0.9, 0.99])[["cuda_event_us"]]

,cuda_event_us
count,100.000000
mean,1234.907521
std,43.728412
min,1211.071968
50%,1220.607996
90%,1276.796818
99%,1385.494994
max,1504.992008


In [10]:
timing_summary = pd.DataFrame(
    {
        "measurement": [
            "Python call without synchronization",
            "Python call including synchronization",
            "CUDA events",
        ],
        "median_us": [
            host_timings["enqueue_us"].median(),
            host_timings["synchronized_us"].median(),
            event_timings["cuda_event_us"].median(),
        ],
        "minimum_us": [
            host_timings["enqueue_us"].min(),
            host_timings["synchronized_us"].min(),
            event_timings["cuda_event_us"].min(),
        ],
        "p90_us": [
            host_timings["enqueue_us"].quantile(0.90),
            host_timings["synchronized_us"].quantile(0.90),
            event_timings["cuda_event_us"].quantile(0.90),
        ],
    }
)

timing_summary

,measurement,median_us,minimum_us,p90_us
0,Python call without synchronization,8.040000,7.488000,10.081600
1,Python call including synchronization,1225.524000,1213.588000,1333.992800
2,CUDA events,1220.607996,1211.071968,1276.796818


## Observations

TODO: Record detection output and measurements only from the current environment.

## Explanation

TODO: Explain launch, execution, and synchronization costs separately.

## Connection to LLMs

LLM kernels rely on CUDA's execution hierarchy, asynchronous launches, and efficient batches of parallel work.

## Further Exploration

TODO: Compare cold-start, warmed-up, synchronized, and unsynchronized timing.